# Democrático — la pareja no se pone de acuerdo en el destino

> **Clasificación: multiagente débil.** Cada agente opina de forma independiente (eso sí es autonomía real), pero el escrutinio final sigue una receta fija mía, no una negociación dinámica entre los agentes.

**Por qué este patrón aquí:** antes de planificar nada, dos personas con preferencias distintas tienen que decidir juntas qué tipo de viaje quieren. No hay un jefe que decida — se vota.

**Nota sobre `Process.consensual`:** si buscas en foros o tutoriales antiguos, verás referencias a un tercer valor de `process` llamado `consensual`, pensado exactamente para este patrón. Estuvo planeado, pero **fue retirado de la documentación oficial de CrewAI** y no es estable — no lo uses en código real. Por eso este ejemplo simula la votación a mano con tasks y un escrutinio.

In [3]:
!uv pip install -r requirements.txt --quiet

In [4]:
from dotenv import load_dotenv

load_dotenv()

import nest_asyncio
nest_asyncio.apply()

In [6]:
from crewai import Agent, Task, Crew, Process

opciones_viaje = [
    "Islandia: naturaleza extrema, auroras boreales, glaciares",
    "Tailandia: playas, templos, gastronomía callejera",
    "Marruecos: desierto, zocos, arquitectura histórica",
]

viajero_1 = Agent(
    role="Preferencias de Viajero 1",
    goal="Evaluar destinos según el gusto por la naturaleza y el frío",
    backstory="Para ti, lo importante son los paisajes extremos y desconectar del calor.",
)
viajero_2 = Agent(
    role="Preferencias de Viajero 2",
    goal="Evaluar destinos según el gusto por la fotografía y los fenómenos naturales",
    backstory="Para ti, lo importante es traerte fotos que nadie más tiene.",
)
amigo_consultado = Agent(
    role="Amigo que ya viajó a los 3 destinos",
    goal="Dar una opinión informada basada en experiencia previa",
    backstory="Has estado en los 3 destinos y tienes opiniones formadas.",
)

votantes = [viajero_1, viajero_2, amigo_consultado]
vote_tasks = []
for votante in votantes:
    t = Task(
        description=(
            f"De estas 3 opciones de destino:\n" + "\n".join(opciones_viaje) +
            "\n\nElige UNA y justifica tu elección en 1-2 frases, según tu propia perspectiva."
        ),
        expected_output="El destino elegido y una justificación breve.",
        agent=votante,
    )
    vote_tasks.append(t)

escrutador = Agent(
    role="Escrutador",
    goal="Contar los votos y anunciar el destino ganador sin opinar",
    backstory="Tu único trabajo es contar votos con exactitud y sin favoritismo.",
)
escrutinio_task = Task(
    description="Cuenta los votos de las 3 evaluaciones anteriores y anuncia qué destino ganó por mayoría (o si hay empate).",
    expected_output="Recuento de votos por destino, y el destino ganador (o empate, con los destinos empatados).",
    agent=escrutador,
    context=vote_tasks,
)

crew = Crew(
    agents=votantes + [escrutador],
    tasks=vote_tasks + [escrutinio_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async()
print(result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c786e4e5-2fef-41cc-8f4a-11ef5859f691                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: De estas 3 opciones de destino:                                                                          │
│  Islandia: naturaleza extrema, auroras boreales, glaciares                                                      │
│  Tailandia: playas, templos, gastronomía callejera                                                              │
│  Marruecos: desierto, zocos, arquitectura histórica                                                             │
│                                                                                                                 │
│  Elige UNA y justifica tu elección en 1-2 frases, según tu propia perspectiva.                                  │
│  ID: e90a8c92-eddd-4667-92d4-bfda826f6524                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Preferencias de Viajero 1                                                                               │
│                                                                                                                 │
│  Task: De estas 3 opciones de destino:                                                                          │
│  Islandia: naturaleza extrema, auroras boreales, glaciares                                                      │
│  Tailandia: playas, templos, gastronomía callejera                                                              │
│  Marruecos: desierto, zocos, arquitectura histórica                                                             │
│                                                                                                                 │
│  Elige UNA y justifica tu elección en 1-2 frases, según tu propia perspectiva.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Preferencias de Viajero 1                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Elijo Islandia porque ofrece paisajes extremos como glaciares y auroras boreales, ideales para desconectar     │
│  del calor y sumergirse en la naturaleza más pura y fría.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Escrutador                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Recuento de votos por destino:                                                                                 │
│  - Islandia: 2 votos                                                                                            │
│  - Tailandia: 1 voto                                                                                            │
│                                                                                                                 │
│  Destino ganador:                                                                                               │
│  Islandia ganó por mayoría con 2 votos.                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Cuenta los votos de las 3 evaluaciones anteriores y anuncia qué destino ganó por mayoría (o si hay       │
│  empate).                                                                                                       │
│  Agent: Escrutador                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c786e4e5-2fef-41cc-8f4a-11ef5859f691                                                                       │
│  Final Output: Recuento de votos por destino:                                                                   │
│  - Islandia: 2 votos                                                                                            │
│  - Tailandia: 1 voto                                                                                            │
│                                                                                                                 │
│  Destino ganador:                                                                                               │
│  Islandia ganó por mayoría con 2 votos.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Recuento de votos por destino:
- Islandia: 2 votos
- Tailandia: 1 voto

Destino ganador:
Islandia ganó por mayoría con 2 votos.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Lo que define el patrón aquí:** la decisión de *qué planificar* — no cómo planificarlo — se reparte entre 3 voces sin jerarquía. Una vez hay un destino ganador, podrías encadenar cualquiera de los 6 patrones anteriores para planificar el viaje en sí.